# Astra 200M — V3 Distillation Notebook (Gemini 3.5 Flash-Lite)

Generates a fresh teacher corpus and trains the student on it, end to end, in one Colab
runtime.

**This is the notebook to run.** It supersedes `astra_v1_distillation_v2.ipynb`.

## Why v3 exists

v2 worked, but three things were wrong in ways that only showed up on a real run:

1. **v2 will overwrite a salvaged corpus.** Its Stage 3 re-splits with per-category rounding
   (533/6/23 becomes 529/10/23) and rewrites `manifest.json` in its own schema, dropping
   `records_in`, `rejected`, `repaired` and `preflight`. v3 writes to
   `datasets/distillation_v3/` and never reads the v2 directory.

2. **The class distribution is badly skewed.** v2 produced 376 suspicious / 116 malicious /
   70 normal — 67% suspicious. A model trained on that flags everything. Reweighting the
   prompt categories does *not* fix it: 30% of v2's categories were normal-leaning and only
   12% of rows came out normal, because the teacher relabels "Normal behavior" scenarios as
   suspicious. v3 caps any one class at 45% of the corpus after generation, where it is
   measurable, and records what it dropped.

3. **v2 could not train on the free Colab GPU.** `torch.amp.autocast` on bfloat16 and
   `numpy` on a bf16 tensor are both unsupported on a T4, so v2 died at step 0 with a
   `CalledProcessError` that hid the cause. Fixed in `training/gpu_train.py`; v3 also
   degrades to fp16 and streams subprocess output instead of swallowing it.

## Cost and time

`TARGET_SAMPLES` teacher calls at concurrency 8, then two training runs. The v2 run of 5000
samples took roughly 90 minutes end to end on a T4. Raise `TARGET_SAMPLES` for a better
model — see the note at Stage 1.


## Stage 0a — Teacher client

Reads the key from **Colab Secrets**, never from a literal. `Runtime → Secrets → + Add new
secret`, name `TEACHER_API_KEY`. The cell refuses to run without it and tells you where to
put it, rather than falling back to a hardcoded string.


In [ ]:
!pip install -q google-genai

import os, json, time, random, hashlib, shutil, threading
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from google import genai
from google.genai import types

TEACHER_MODEL = 'gemini-3.5-flash-lite'   # GA 2026-07-21; supersedes 3.1-flash-lite


def resolve_api_key():
    key = os.environ.get('TEACHER_API_KEY')
    if key:
        return key
    try:
        from google.colab import userdata
        return userdata.get('TEACHER_API_KEY')
    except Exception:
        return None


api_key = resolve_api_key()
assert api_key, (
    'Set TEACHER_API_KEY in Colab Secrets: Runtime -> Secrets -> + Add new secret. '
    'Do not paste the key into a cell - it ends up in the notebook, and Colab '
    'notebooks are shareable.')
client = genai.Client(api_key=api_key)

try:
    probe = client.models.generate_content(model=TEACHER_MODEL,
                                          contents='Reply with one word: ready')
    print('teacher reachable ->', probe.text.strip()[:40])
except Exception as exc:
    raise SystemExit('Teacher model %s is unreachable: %s' % (TEACHER_MODEL, exc))

print('teacher model :', TEACHER_MODEL)
print('client        : initialised')

## Stage 0b — Repository, Drive, directories

Clones on first run; Colab resets `/content` on reconnect, so it repeats. Everything is
mirrored to Drive as it is produced, because a reconnect otherwise loses the corpus and the
GPU hours spent on it.

The Drive mirror includes the **tokenizer**, not just the data and configs. `.gitignore`
excludes `tokenizer/artifacts/`, so a fresh clone has no tokenizer and every emitted config
is a path to a file that is not there.


In [ ]:
import os, subprocess

REPO = '/content/astra'
REPO_URL = 'https://github.com/anoneurx/astra.git'
DATA_DIR = 'datasets/distillation_v3'
# Checkpoints live outside the repo so `git status` never sees them. Overridable so this
# notebook also runs outside Colab, where /content may not be writable.
RUNS = os.environ.get('ASTRA_RUNS', '/content/runs')
os.makedirs(RUNS, exist_ok=True)
DRIVE_ROOT = '/content/drive/MyDrive/astra/distillation_v3'

if not os.path.isdir(os.path.join(REPO, 'tools')):
    subprocess.run('git clone --depth 1 %s %s' % (REPO_URL, REPO), shell=True, check=True)
os.chdir(REPO)
assert os.path.isdir(os.path.join(REPO, 'tools')), 'astra clone missing'

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    DRIVE_OK = os.path.isdir(DRIVE_ROOT)
except Exception as exc:
    DRIVE_OK = False
    print('Google Drive unavailable (%s) - artifacts stay under /content only.' % exc)

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs('/content/runs', exist_ok=True)

print('repo        :', os.getcwd())
print('data dir    :', os.path.abspath(DATA_DIR))
print('drive mirror:', DRIVE_ROOT if DRIVE_OK else 'DISABLED')

## Stage 1 — Teacher configuration

`TARGET_SAMPLES` is raised to 8000. The salvaged 533-row run trained to a plateau at
perplexity 17.9 and generated word salad: 8.1M parameters against 129k tokens is not enough
signal, and it **plateaued rather than memorising** — more steps at that data volume would
not have helped. Volume is the lever.

Categories are the prompt archetypes. Note they do **not** map 1:1 onto `classification` —
the teacher decides the label — so the weights here are not what balances the corpus. That
is handled at Stage 3.


In [ ]:
import os, random

os.chdir('/content/astra')

SEED = 20051
TARGET_SAMPLES = 8000
CONCURRENCY = 8
VAL_FRAC, EVAL_FRAC = 0.01, 0.04

# Cap on any single classification, as a share of the corpus. The v2 run came out 67%
# suspicious / 21% malicious / 12% normal. Enforced at Stage 3, where it is measurable.
MAX_CLASS_SHARE = 0.45
# Floor per class, so a cap can never starve a class out of the training split.
MIN_PER_CLASS = 40
MIRROR_EVERY = 50
MAX_ATTEMPTS = 5
MAX_CONSECUTIVE_FAILURES = 25
MAX_OUTPUT_TOKENS = 1200
TEMPERATURE = 0.9
WINDOW = CONCURRENCY * 4        # jobs kept in flight; the rest wait in a local deque

CLASSIFICATIONS = ('normal', 'suspicious', 'malicious', 'unknown')
RISK_LEVELS = ('INFO', 'LOW', 'MEDIUM', 'HIGH', 'CRITICAL')
HARDNESS_BY_RISK = {'INFO': 5, 'LOW': 20, 'MEDIUM': 45, 'HIGH': 70, 'CRITICAL': 90}

ENVIRONMENTS = [
    'a public cloud Kubernetes cluster',
    'a regional hospital network with legacy medical devices',
    'a university campus network during registration week',
    'a retail payment processing environment',
    'a small manufacturing plant with an industrial control segment',
    'a corporate datacentre with a corporate VPN',
    'a managed service provider moving between customer tenants',
    'a remote-first software company behind a zero-trust access proxy',
    'a freight and logistics operator with legacy telematics units',
    'a municipal water utility with remote monitoring stations',
    'a law firm with an externally hosted document management system',
    'a film studio with a distributed render farm and shared storage',
]

LENSES = [
    'one quiet off-hours login that is genuinely routine',
    'a burst of failed logins from many distinct source addresses',
    'an internal host initiating outbound traffic it has never used before',
    'a legitimate change ticket that produces a signature identical to an attack',
    'a scheduled backup job whose behaviour only looks wrong in isolation',
    'a service account whose permissions were widened last week',
    'a new device on the network with no asset record',
    'DNS lookups unusual for the requesting host but normal for the domain',
    'a user who simply works unusual hours',
    'two unrelated alerts that correlate into one real incident',
    'a misconfigured sensor generating high-volume duplicate alerts',
    'an expired certificate on a critical path',
    'a patch window producing a large but legitimate change footprint',
    'a third party with a risky integration but a contractual agreement',
    'a container image pull pattern that differs from the cluster baseline',
    'a known-noisy monitoring rule that fires on a real problem today',
    'an account whose second factor was disabled by an administrator',
    'a firewall rule that was widened while an incident was still open',
    'a backup that grew several times larger overnight',
    'a certificate that was reissued shortly before it expired',
]

# The grid has to stay larger than the per-category quota, otherwise the prompt walk cycles
# and every job past the cycle point re-asks a framing the teacher has already answered.
# 8 x 16 x 8 = 1024 framings against a 1000-sample quota.
TIMINGS = [
    'during a normal working day',
    'in the small hours of the morning',
    'over a holiday weekend with only a skeleton crew on call',
    'in the fifteen minutes after a deployment finished',
    'during the weekly backup window',
    'shortly after a firewall rule change',
    'at the end of a billing cycle',
    'while a vendor was performing scheduled maintenance',
]

CATEGORIES = [
    ('Normal behavior', 0.20,
     'Write a normal event stream containing nothing anomalous at all.'),
    ('Suspicious behavior', 0.20,
     'Write a scenario containing behaviour that is concerning but does not confirm an attack. '
     'classification must be "suspicious" or "unknown", never "malicious".'),
    ('Early-warning sequences', 0.20,
     'Write a chronological stream of 5 to 15 events that starts routine and drifts. '
     'early_warning must be true and the early signal must appear before the decisive event.'),
    ('False positives', 0.10,
     'Write a scenario that reads as an attack but has a documented benign explanation. '
     'classification must be "normal" or "suspicious".'),
    ('Multi-signal correlation', 0.10,
     'Write a scenario where no single event is malicious but the combination is. early_warning '
     'must be true and evidence must list every contributing signal.'),
    ('Authentication', 0.05,
     'Write an authentication scenario mixing routine, unusual and suspicious login patterns.'),
    ('Network', 0.05,
     'Write a network event sequence covering DNS resolution, connection setup and traffic anomalies.'),
    ('Process behavior', 0.05,
     'Write a host process sequence - process tree, user, parent, hashes, privilege changes - '
     'that needs context to judge.'),
    ('Vulnerability / code security', 0.05,
     'Write a code or configuration defect scenario. Quote only the flawed fragment, and focus '
     'on detection and remediation.'),
]

REQUIRED_FIELDS = ('input', 'classification', 'risk', 'early_warning',
                   'evidence', 'analysis', 'recommended_action', 'confidence')

RESPONSE_SCHEMA = {
    'type': 'object',
    'properties': {
        'input': {'type': 'string'},
        'classification': {'type': 'string', 'enum': list(CLASSIFICATIONS)},
        'risk': {'type': 'string', 'enum': list(RISK_LEVELS)},
        'early_warning': {'type': 'boolean'},
        'evidence': {'type': 'array', 'items': {'type': 'string'}},
        'analysis': {'type': 'string'},
        'recommended_action': {'type': 'string'},
        'confidence': {'type': 'number'},
    },
    'required': list(REQUIRED_FIELDS),
}


def quota_plan(total, weights):
    """Largest-remainder apportionment: integer counts that sum to exactly *total*."""
    raw = [(name, w * total) for name, w in weights]
    counts = {name: int(frac) for name, frac in raw}
    order = sorted(range(len(raw)),
                   key=lambda i: (-(raw[i][1] - counts[raw[i][0]]), i))
    for i in order[: total - sum(counts.values())]:
        counts[raw[i][0]] += 1
    assert sum(counts.values()) == total
    return counts


QUOTAS = quota_plan(TARGET_SAMPLES, [(n, w) for n, w, _ in CATEGORIES])
GRID = len(ENVIRONMENTS) * len(LENSES) * len(TIMINGS)

print('quota plan (sum=%d):' % sum(QUOTAS.values()))
for name, _, _ in CATEGORIES:
    print('  %-30s %5d  %5.1f%%' % (name, QUOTAS[name], 100.0 * QUOTAS[name] / TARGET_SAMPLES))
print()
print('prompt framings per category : %d (%d env x %d lens x %d timing)'
      % (GRID, len(ENVIRONMENTS), len(LENSES), len(TIMINGS)))
print('largest per-category quota   : %d' % max(QUOTAS.values()))
assert GRID > max(QUOTAS.values()), \
    'grid smaller than the quota: prompts would cycle and re-ask answered framings'
print('grid exceeds every quota     : OK')

## Stage 2a — Persistence and resume state

Writes `progress.json` next to the records and mirrors to Drive every `MIRROR_EVERY`
accepted rows. If the runtime dies, re-running this cell resumes rather than re-paying for
rows it already has.


In [ ]:
import os, json, shutil, subprocess

REPO = '/content/astra'
if not os.path.isdir(os.path.join(REPO, 'tools')):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git %s' % REPO,
                   shell=True, check=True)
os.chdir(REPO)

DATA_DIR = 'datasets/distillation_v3'
RECORDS = os.path.join(DATA_DIR, 'records.jsonl')
PROGRESS = os.path.join(DATA_DIR, 'progress.json')
DRIVE_ROOT = '/content/drive/MyDrive/astra/distillation_v3'

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_OK = os.path.isdir(DRIVE_ROOT)
except Exception:
    DRIVE_OK = False

# drive name -> path relative to the repo root. The chat/ and alerts/ splits use the same
# three filenames, so a flat mirror would have alerts silently overwrite chat. Mirror them
# under their own subdirectories, and put the two configs in configs/ where the trainer reads
# them. progress.json is mirrored too: without it a resume re-tries framings it already used.
MIRRORED = {
    'records.jsonl': RECORDS,
    'progress.json': PROGRESS,
    'manifest.json': os.path.join(DATA_DIR, 'manifest.json'),
    'chat/train.txt': os.path.join(DATA_DIR, 'chat/train.txt'),
    'chat/val.txt': os.path.join(DATA_DIR, 'chat/val.txt'),
    'chat/eval.txt': os.path.join(DATA_DIR, 'chat/eval.txt'),
    'alerts/train.txt': os.path.join(DATA_DIR, 'alerts/train.txt'),
    'alerts/val.txt': os.path.join(DATA_DIR, 'alerts/val.txt'),
    'alerts/eval.txt': os.path.join(DATA_DIR, 'alerts/eval.txt'),
    'astra_distill_v3_chat.json': 'configs/astra_distill_v3_chat.json',
    'astra_distill_v3_alerts.json': 'configs/astra_distill_v3_alerts.json',
}

restored = []
if DRIVE_OK and os.path.isdir(DRIVE_ROOT):
    for name, rel in sorted(MIRRORED.items()):
        src, dst = os.path.join(DRIVE_ROOT, name), rel
        if not os.path.exists(src):
            continue
        if os.path.exists(dst) and os.path.getsize(dst) >= os.path.getsize(src):
            continue
        parent = os.path.dirname(dst)
        if parent:
            os.makedirs(parent, exist_ok=True)
        shutil.copy2(src, dst)
        restored.append(name)
    if restored:
        print('restored from Drive:', ', '.join(restored))

records = []
unreadable = 0
if os.path.exists(RECORDS):
    with open(RECORDS, encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                unreadable += 1
    print('resume: %d records on disk, %d unreadable lines' % (len(records), unreadable))
else:
    print('resume: no records.jsonl yet - starting fresh')

CATEGORIES = ['Normal behavior', 'Suspicious behavior', 'Early-warning sequences',
              'False positives', 'Multi-signal correlation', 'Authentication',
              'Network', 'Process behavior', 'Vulnerability / code security']

state = {c: 0 for c in CATEGORIES}
seen_keys = set()
for rec in records:
    if rec.get('dedup_key'):
        seen_keys.add(rec['dedup_key'])
    if rec.get('category') in state:
        state[rec['category']] += 1

print('per-category counts already on disk:')
for name in CATEGORIES:
    print('  %-30s %5d' % (name, state[name]))

## Stage 2b — Generation loop

Concurrent teacher calls against a fixed environment x lens x timing grid, with
quota-per-category, retry on transient failure, and a degeneration guard: a teacher that
loops will emit the same six events fifty times, which is valid JSON, passes every enum
check, and is worse than useless as training data. Any 60-character span occurring three or
more times in a row is rejected.

Nothing here is new relative to v2 — this is the part that already worked.


In [ ]:
import os, json, time, random, hashlib, shutil, subprocess, threading
from collections import deque
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from google import genai
from google.genai import types

REPO = '/content/astra'
if not os.path.isdir(os.path.join(REPO, 'tools')):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git %s' % REPO,
                   shell=True, check=True)
os.chdir(REPO)

TEACHER_MODEL = 'gemini-3.5-flash-lite'
DATA_DIR = 'datasets/distillation_v3'
RECORDS = os.path.join(DATA_DIR, 'records.jsonl')
PROGRESS = os.path.join(DATA_DIR, 'progress.json')
DRIVE_ROOT = '/content/drive/MyDrive/astra/distillation_v3'
os.makedirs(DATA_DIR, exist_ok=True)

# Rebuild the client if Stage 0a has not run in this kernel, so this cell is runnable alone.
if 'client' not in globals():
    _key = os.environ.get('TEACHER_API_KEY')
    if not _key:
        try:
            from google.colab import userdata
            _key = userdata.get('TEACHER_API_KEY')
        except Exception:
            _key = None
    assert _key, 'Set TEACHER_API_KEY in Colab Secrets (or the environment) first.'
    client = genai.Client(api_key=_key)
    print('client rebuilt (Stage 0a had not run in this kernel)')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_OK = os.path.isdir(DRIVE_ROOT)
except Exception:
    DRIVE_OK = False


def mirror_to_drive():
    if not DRIVE_OK:
        return
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    for name in ('records.jsonl', 'manifest.json'):
        src = os.path.join(DATA_DIR, name)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(DRIVE_ROOT, name))


SEED = 20051
# Must match Stage 1. This cell is runnable on its own, so it repeats the constant
# rather than depending on earlier cells - which is how the two drifted apart.
TARGET_SAMPLES = 8000
CONCURRENCY = 8
MIRROR_EVERY = 50
MAX_ATTEMPTS = 5
MAX_CONSECUTIVE_FAILURES = 25
MAX_OUTPUT_TOKENS = 1200
TEMPERATURE = 0.9
WINDOW = CONCURRENCY * 4        # jobs kept in flight; the rest wait in a local deque
# A job dropped as a duplicate or a schema reject leaves its category short of quota, so each
# category is asked for a little extra. Anything accepted past the quota is discarded and
# counted as `overflow`, which keeps the realized distribution exactly on plan.
DEDUP_HEADROOM = 0.05

CLASSIFICATIONS = ('normal', 'suspicious', 'malicious', 'unknown')
RISK_LEVELS = ('INFO', 'LOW', 'MEDIUM', 'HIGH', 'CRITICAL')
HARDNESS_BY_RISK = {'INFO': 5, 'LOW': 20, 'MEDIUM': 45, 'HIGH': 70, 'CRITICAL': 90}
REQUIRED_FIELDS = ('input', 'classification', 'risk', 'early_warning',
                   'evidence', 'analysis', 'recommended_action', 'confidence')
MIN_INPUT_CHARS, MIN_ANALYSIS_CHARS = 120, 40
# These caps bound a row. They do NOT make an exchange fit the 256-token block, and the
# comment that used to claim they did was wrong. Measured on the salvaged corpus at
# 3.17 chars/token, a 256-token block holds ~812 chars for the input AND the target
# together, while 480 + ~500 leaves ~980. 31% of blocks still overflow. Tightening input
# alone cannot fix it - at input<=240 it is still 6.6% over, because the target is the
# binding side. What the cap does buy is a worst case: max block fell 421 -> 380 tokens
# versus V1's 1200 cap, and a runaway teacher is rejected at generation time rather than
# discovered in the audit. The pre-flight cell reports the real distribution; read it
# before assuming a corpus fits.
MAX_INPUT_CHARS, MAX_ANALYSIS_CHARS = 480, 420
# A teacher that loops will happily emit the same six events fifty times. That is valid JSON,
# passes every enum check and is worse than useless as training data, so reject it: if any
# 60-character span of the row occurs three or more times, the generation degenerated.
# Calibrated against every corpus in datasets/ - zero false positives over 7958 chat blocks.
REPEAT_ANCHOR, MAX_REPEATS = 60, 3
# A flag the model can actually learn: anything actionable is an early warning.
EARLY_WARNING_CLASSIFICATIONS = ('suspicious', 'malicious')
MAX_CONFIDENCE = 0.97
MIN_ACTION_CHARS = 40
# Phrases that assert there is nothing to do. Measured on V1: 122 of 1002 targets were under
# 40 characters and the single most common was "No action required." 93 times.
NO_OP_ACTIONS = ('no action required', 'no action needed', 'none', 'no action',
                 'no further action', 'nothing', 'n/a', 'not applicable')

SYSTEM_PROMPT = """You are the teacher model for Astra, a small cybersecurity research model trained for defensive security analysis.
Generate realistic, self-consistent security telemetry and label it so a student model can learn:
anomaly detection, suspicious-behaviour detection, early warning, security-event sequence reasoning,
vulnerability identification, risk assessment, evidence-grounded reasoning, and defensive response.

Rules you must follow:
- Defensive framing only. Describe what an analyst observes. Never give exploitation steps,
  payloads, credential lists, or instructions for unauthorised access.
- Ground every label in the telemetry you wrote. Evidence entries must be verbatim fragments
  of the input, not paraphrases.
- classification is exactly one of: normal, suspicious, malicious, unknown.
- risk is exactly one of: INFO, LOW, MEDIUM, HIGH, CRITICAL - UPPERCASE.
- early_warning MUST be true when classification is suspicious or malicious, and false when it
  is normal or unknown. This is enforced on every record, not a suggestion.
- recommended_action must be a specific, actionable instruction of at least 40 characters that
  names what to check, isolate, rotate or review. Never answer "No action required", "None",
  or any other statement that nothing needs to happen. Even a benign scenario gets a concrete
  monitoring or verification step.
- Write input as a SINGLE line. Use " | " between events. No newlines, no numbering, no
  "Chronological Events:" headers, no markdown.
- Keep input under 480 characters and analysis under 420 characters. Write six to eight
  distinct events and never repeat one. These are row-size bounds, not a block-fit
  guarantee: the target needs room too, and the pre-flight cell reports the real
  block-length distribution.
- Evidence entries must be verbatim fragments of the input, so the student can learn to cite.
- confidence is a number strictly between 0.5 and 0.97. Do not answer 1.0; nothing is certain.
- Vary hosts, users, ports, timestamps and protocols. Do not reuse the same scenario twice.
- Return ONLY a JSON object: no prose, no markdown fence."""

ENVIRONMENTS = [
    'a public cloud Kubernetes cluster',
    'a regional hospital network with legacy medical devices',
    'a university campus network during registration week',
    'a retail payment processing environment',
    'a small manufacturing plant with an industrial control segment',
    'a corporate datacentre with a corporate VPN',
    'a managed service provider moving between customer tenants',
    'a remote-first software company behind a zero-trust access proxy',
    'a freight and logistics operator with legacy telematics units',
    'a municipal water utility with remote monitoring stations',
    'a law firm with an externally hosted document management system',
    'a film studio with a distributed render farm and shared storage',
]

LENSES = [
    'one quiet off-hours login that is genuinely routine',
    'a burst of failed logins from many distinct source addresses',
    'an internal host initiating outbound traffic it has never used before',
    'a legitimate change ticket that produces a signature identical to an attack',
    'a scheduled backup job whose behaviour only looks wrong in isolation',
    'a service account whose permissions were widened last week',
    'a new device on the network with no asset record',
    'DNS lookups unusual for the requesting host but normal for the domain',
    'a user who simply works unusual hours',
    'two unrelated alerts that correlate into one real incident',
    'a misconfigured sensor generating high-volume duplicate alerts',
    'an expired certificate on a critical path',
    'a patch window producing a large but legitimate change footprint',
    'a third party with a risky integration but a contractual agreement',
    'a container image pull pattern that differs from the cluster baseline',
    'a known-noisy monitoring rule that fires on a real problem today',
    'an account whose second factor was disabled by an administrator',
    'a firewall rule that was widened while an incident was still open',
    'a backup that grew several times larger overnight',
    'a certificate that was reissued shortly before it expired',
]

# The grid has to stay larger than the per-category quota, otherwise the prompt walk cycles
# and every job past the cycle point re-asks a framing the teacher has already answered.
# At TARGET_SAMPLES = 8000 the largest quota is 1600 and the grid is 12 x 20 x 8 = 1920.
# The assert below recomputes both from the lists, so changing either is caught here.
TIMINGS = [
    'during a normal working day',
    'in the small hours of the morning',
    'over a holiday weekend with only a skeleton crew on call',
    'in the fifteen minutes after a deployment finished',
    'during the weekly backup window',
    'shortly after a firewall rule change',
    'at the end of a billing cycle',
    'while a vendor was performing scheduled maintenance',
]

CATEGORIES = [
    ('Normal behavior', 0.20,
     'Write a normal event stream containing nothing anomalous at all.'),
    ('Suspicious behavior', 0.20,
     'Write a scenario containing behaviour that is concerning but does not confirm an attack. '
     'classification must be "suspicious" or "unknown", never "malicious".'),
    ('Early-warning sequences', 0.20,
     'Write a chronological stream of 5 to 15 events that starts routine and drifts. '
     'early_warning must be true and the early signal must appear before the decisive event.'),
    ('False positives', 0.10,
     'Write a scenario that reads as an attack but has a documented benign explanation. '
     'classification must be "normal" or "suspicious".'),
    ('Multi-signal correlation', 0.10,
     'Write a scenario where no single event is malicious but the combination is. early_warning '
     'must be true and evidence must list every contributing signal.'),
    ('Authentication', 0.05,
     'Write an authentication scenario mixing routine, unusual and suspicious login patterns.'),
    ('Network', 0.05,
     'Write a network event sequence covering DNS resolution, connection setup and traffic anomalies.'),
    ('Process behavior', 0.05,
     'Write a host process sequence - process tree, user, parent, hashes, privilege changes - '
     'that needs context to judge.'),
    ('Vulnerability / code security', 0.05,
     'Write a code or configuration defect scenario. Quote only the flawed fragment, and focus '
     'on detection and remediation.'),
]

RESPONSE_SCHEMA = {
    'type': 'object',
    'properties': {
        'input': {'type': 'string'},
        'classification': {'type': 'string', 'enum': list(CLASSIFICATIONS)},
        'risk': {'type': 'string', 'enum': list(RISK_LEVELS)},
        'early_warning': {'type': 'boolean'},
        'evidence': {'type': 'array', 'items': {'type': 'string'}},
        'analysis': {'type': 'string'},
        'recommended_action': {'type': 'string'},
        'confidence': {'type': 'number'},
    },
    'required': list(REQUIRED_FIELDS),
}


def quota_plan(total, weights):
    raw = [(name, w * total) for name, w in weights]
    counts = {name: int(frac) for name, frac in raw}
    order = sorted(range(len(raw)),
                   key=lambda i: (-(raw[i][1] - counts[raw[i][0]]), i))
    for i in order[: total - sum(counts.values())]:
        counts[raw[i][0]] += 1
    assert sum(counts.values()) == total
    return counts


def strip_fences(text):
    """Recover the JSON object from a reply that arrived wrapped in a markdown fence.

    `response_mime_type` should prevent fences entirely, but the schema-less retry path and
    a model that ignores the mime type both fall back to raw text, and V1 wrote fenced
    replies straight into the corpus.
    """
    text = str(text).strip()
    if not text.startswith('```'):
        return text
    head, sep, rest = text.partition('\n')
    if sep:
        text = rest
    else:
        text = text[3:].lstrip()
        if text[:4].lower() == 'json':
            text = text[4:].lstrip()
    if text.endswith('```'):
        text = text[:-3]
    return text.strip()


def one_line(text, sep=' | '):
    """Collapse a multi-line teacher field onto one line.

    The chat framing is `You: <one line>\\nAstra: <one line>`; a newline inside the user turn
    teaches the model that it has already answered half the block.
    """
    return sep.join(p.strip() for p in str(text).splitlines() if p.strip())


def dedup_key(text):
    norm = ' '.join(str(text).lower().split())
    return 'distill:' + hashlib.sha256(norm.encode('utf-8')).hexdigest()[:16]


def is_no_op(action):
    """True when the action asserts that nothing needs to happen.

    V1 emitted "No action required." 93 times, "None." 18 times and several padded variants.
    Training on those teaches the model that the useful answer is sometimes nothing at all.
    """
    body = action.strip().rstrip('.!? ').lower()
    if body in NO_OP_ACTIONS:
        return True
    # also catch a padded form such as "No action required. This is standard telemetry."
    return any(body.startswith(phrase) and len(body) < len(phrase) + 90
               for phrase in NO_OP_ACTIONS)


def is_repetitive(text, anchor=REPEAT_ANCHOR, max_repeats=MAX_REPEATS, window=MAX_INPUT_CHARS):
    """True when a long span of the text occurs repeatedly, i.e. the teacher looped.

    Samples three fixed offsets and counts occurrences of each anchor. Three substring counts
    is enough signal, is alignment-independent (a shingle or word-ratio test is not), and does
    not fire on natural repetition of common words in long prose.
    """
    head = text[:window]
    if len(head) < 4 * anchor:
        return False
    for off in (0, len(head) // 2, len(head) - 2 * anchor):
        frag = head[off:off + anchor]
        if len(frag) < anchor or not frag.strip():
            continue
        if head.count(frag) >= max_repeats:
            return True
    return False


def validate_record(raw):
    """Return (clean_record, None) or (None, reason). Never raises."""
    if not isinstance(raw, dict):
        return None, 'not_an_object'
    missing = [f for f in REQUIRED_FIELDS if f not in raw]
    if missing:
        return None, 'missing:' + ','.join(missing)

    text = one_line(raw['input'])
    if len(text) < MIN_INPUT_CHARS:
        return None, 'input_too_short'
    if len(text) > MAX_INPUT_CHARS:
        return None, 'input_too_long'
    if is_repetitive(text):
        return None, 'input_repetitive'

    classification = str(raw['classification']).strip().lower()
    if classification not in CLASSIFICATIONS:
        return None, 'bad_classification'

    risk = str(raw['risk']).strip().upper()
    if risk not in RISK_LEVELS:
        return None, 'bad_risk'

    evidence = raw['evidence']
    if isinstance(evidence, str):
        evidence = [evidence]
    if not isinstance(evidence, list):
        return None, 'evidence_not_list'
    evidence = [one_line(e, sep=' ') for e in evidence if str(e).strip()]
    if not evidence:
        return None, 'evidence_empty'

    analysis = one_line(raw['analysis'], sep=' ')
    action = one_line(raw['recommended_action'], sep=' ')
    if len(analysis) < MIN_ANALYSIS_CHARS:
        return None, 'analysis_too_short'
    if len(analysis) > MAX_ANALYSIS_CHARS or len(action) > MAX_ANALYSIS_CHARS:
        return None, 'text_too_long'
    if is_repetitive(analysis):
        return None, 'analysis_repetitive'
    if not action:
        return None, 'no_recommended_action'
    # No-op is checked before length: V1 shipped 54-character no-ops such as "No action
    # required. This is standard system telemetry." that sail past a length gate, and the
    # specific diagnosis is the one worth reporting.
    if is_no_op(action):
        return None, 'action_is_no_op'
    if len(action) < MIN_ACTION_CHARS:
        return None, 'action_too_short'

    try:
        confidence = float(raw['confidence'])
    except (TypeError, ValueError):
        return None, 'bad_confidence'
    confidence = min(1.0, max(0.0, confidence))
    # V1 pinned 204 of 1002 rows at exactly 1.0. A teacher is not an oracle; a ceiling below
    # 1.0 keeps the distribution informative instead of clustering at the top of the range.
    confidence = min(MAX_CONFIDENCE, confidence)

    early = raw['early_warning']
    if isinstance(early, str):
        early = early.strip().lower() in ('true', 'yes', '1')
    early = bool(early)
    # V1 shipped 74 malicious rows with early_warning=false against 188 true, while every
    # suspicious row was true and 282 of 283 normal rows were false. A flag the model cannot
    # learn is label noise, so the rule is stated once here and enforced on every record.
    if early != (classification in EARLY_WARNING_CLASSIFICATIONS):
        return None, 'early_warning_inconsistent'

    return {
        'input': text,
        'classification': classification,
        'risk': risk,
        'early_warning': early,
        'evidence': evidence,
        'analysis': analysis,
        'recommended_action': action,
        'confidence': confidence,
        'hardness': HARDNESS_BY_RISK[risk],
    }, None


def job_prompt(category_prompt, index):
    env = ENVIRONMENTS[index % len(ENVIRONMENTS)]
    lens = LENSES[(index // len(ENVIRONMENTS)) % len(LENSES)]
    when = TIMINGS[(index // (len(ENVIRONMENTS) * len(LENSES))) % len(TIMINGS)]
    return ('Setting: %s.\n'
            'Focus this example on: %s.\n'
            'Timing: %s.\n'
            '%s\n'
            'Write "input" as concrete telemetry in at most 8 short lines: timestamps, hosts, '
            'users, processes, ports and counts. Be specific and vary every value.\n'
            'Return ONLY the JSON object.' % (env, lens, when, category_prompt))


FATAL_MARKERS = ('unauthenticated', 'permission denied', 'invalid api key',
                 'invalid argument', 'billing', 'quota exceeded for', 'forbidden',
                 'api key not valid')


def is_fatal(exc):
    text = repr(exc).lower()
    return any(marker in text for marker in FATAL_MARKERS)


class Aborted(RuntimeError):
    """Raised in a worker when a sibling hit a fatal error, to stop the queue draining."""


ABORT = threading.Event()

def call_teacher(prompt, use_schema=True):
    """One teacher call with backoff. Returns a dict, or raises when it gives up."""
    last = None
    for attempt in range(MAX_ATTEMPTS):
        if ABORT.is_set():
            raise Aborted('aborted after a fatal error')
        try:
            kwargs = dict(system_instruction=SYSTEM_PROMPT,
                          response_mime_type='application/json',
                          temperature=TEMPERATURE,
                          max_output_tokens=MAX_OUTPUT_TOKENS)
            if use_schema:
                kwargs['response_schema'] = RESPONSE_SCHEMA
            resp = client.models.generate_content(
                model=TEACHER_MODEL, contents=prompt,
                config=types.GenerateContentConfig(**kwargs))
            return json.loads(strip_fences(resp.text))
        except json.JSONDecodeError as exc:
            last = exc
            if not use_schema:
                break
            use_schema = False          # schema may be unsupported; retry without it
        except (TypeError, AttributeError, NameError) as exc:
            last = exc
            if not use_schema:
                raise
            use_schema = False          # SDK-side shape problem; one clean retry
        except Exception as exc:
            last = exc
            if is_fatal(exc):
                raise
            time.sleep(min(45.0, 2 ** attempt) * (0.5 + random.random()))
    raise last


QUOTAS = quota_plan(TARGET_SAMPLES, [(n, w) for n, w, _ in CATEGORIES])
GRID = len(ENVIRONMENTS) * len(LENSES) * len(TIMINGS)
assert GRID > max(QUOTAS.values()), (
    'grid %d is not larger than the largest per-category quota %d: prompts would cycle'
    % (GRID, max(QUOTAS.values())))

def write_progress(cursor, state):
    """Persist the attempt cursor and accepted counts so a resume never re-tries a framing."""
    payload = {'updated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
               'cursor': dict(cursor), 'accepted': dict(state),
               'quotas': dict(QUOTAS), 'seed': SEED}
    tmp = PROGRESS + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as fh:
        json.dump(payload, fh, indent=2)
    os.replace(tmp, PROGRESS)          # atomic: a crash cannot leave a half-written cursor
    if DRIVE_OK:
        try:
            shutil.copy2(PROGRESS, os.path.join(DRIVE_ROOT, os.path.basename(PROGRESS)))
        except Exception as exc:
            print('  progress mirror failed:', exc)

# Two separate counters per category:
#   state[name]  - records actually on disk, rebuilt from records.jsonl (authoritative)
#   cursor[name] - how many distinct grid framings have been *tried*, from progress.json
# The grid index handed to job_prompt must come from the cursor, not from state. A category
# that lost records to a duplicate or a schema reject would otherwise re-request the exact
# same framings on the next run, every one of them a duplicate, and could never backfill.
state = {name: 0 for name, _, _ in CATEGORIES}
seen_keys, written_total = set(), 0
if os.path.exists(RECORDS):
    with open(RECORDS, encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            if rec.get('dedup_key'):
                seen_keys.add(rec['dedup_key'])
            if rec.get('category') in state:
                state[rec['category']] += 1
                written_total += 1

# The cursor is advanced and persisted the moment a job is queued, so a crash mid-run never
# re-spends calls on framings that were already tried. A missing or corrupt file is not fatal:
# the worst case is re-asking for a few framings, and dedup absorbs that.
cursor = {name: state[name] for name, _, _ in CATEGORIES}
if os.path.exists(PROGRESS):
    try:
        with open(PROGRESS, encoding='utf-8') as fh:
            saved = json.load(fh).get('cursor', {})
        for name in cursor:
            cursor[name] = max(cursor[name], int(saved.get(name, 0)))
        print('resumed attempt cursor from', PROGRESS)
    except Exception as exc:
        print('progress file unreadable, rebuilding from records (%r)' % (exc,))

jobs = []
for name, _, prompt in CATEGORIES:
    if state[name] >= QUOTAS[name]:
        continue                      # already at quota: do not spend calls on headroom
    target = QUOTAS[name] + max(1, round(QUOTAS[name] * DEDUP_HEADROOM))
    for _ in range(target - state[name]):
        jobs.append((name, prompt, cursor[name]))
        cursor[name] += 1
random.Random(SEED).shuffle(jobs)
write_progress(cursor, state)

print('resuming at %d records; requesting %d more' % (written_total, len(jobs)))

if not jobs:
    print('quota already met - nothing to do')
else:
    accepted, rejects, duplicates, overflow, consecutive = 0, {}, 0, 0, 0
    examples, last_flush = [], time.time()

    with open(RECORDS, 'a', encoding='utf-8') as sink:
        def flush(mirror=False):
            sink.flush()
            if mirror:
                mirror_to_drive()

        # Keep only a bounded window of jobs in flight instead of queueing all of them.
        # Submitting everything up front means a pool that fails fast keeps draining its own
        # queue faster than the main thread can call cancel(), so a dead API key still burns
        # hundreds of calls, and TARGET_SAMPLES futures sit in memory for the whole run.
        todo, in_flight, stopped = deque(jobs), {}, False

        def submit(pool):
            cat, prompt, idx = todo.popleft()
            in_flight[pool.submit(call_teacher, job_prompt(prompt, idx))] = (cat, idx)

        with ThreadPoolExecutor(max_workers=CONCURRENCY) as pool:
            for _ in range(min(WINDOW, len(todo))):
                submit(pool)

            while in_flight and not stopped:
                for fut in wait(in_flight, return_when=FIRST_COMPLETED).done:
                    cat, idx = in_flight.pop(fut)
                    try:
                        raw = fut.result()
                    except Aborted:
                        stopped = True
                        break
                    except Exception as exc:
                        consecutive += 1
                        reason = 'api:' + type(exc).__name__
                        rejects[reason] = rejects.get(reason, 0) + 1
                        if is_fatal(exc):
                            print('\nFATAL teacher error, aborting: %r' % (exc,))
                            ABORT.set()  # in-flight workers bail without another call
                            stopped = True
                            break
                        if consecutive >= MAX_CONSECUTIVE_FAILURES:
                            print('\ncircuit breaker: %d consecutive failures. '
                                  'Re-run to resume.' % consecutive)
                            ABORT.set()
                            stopped = True
                            break
                        if todo and not ABORT.is_set():
                            submit(pool)
                        continue
                    consecutive = 0

                    rec, reason = validate_record(raw)
                    if rec is None:
                        rejects[reason] = rejects.get(reason, 0) + 1
                        if len(examples) < 5:
                            examples.append((cat, reason, json.dumps(raw)[:180]))
                        if todo and not ABORT.is_set():
                            submit(pool)
                        continue

                    key = dedup_key(rec['input'])
                    if key in seen_keys:
                        duplicates += 1
                        if todo and not ABORT.is_set():
                            submit(pool)
                        continue
                    if state[cat] >= QUOTAS[cat]:
                        overflow += 1
                        if todo and not ABORT.is_set():
                            submit(pool)
                        continue
                    seen_keys.add(key)
                    if todo and not ABORT.is_set():
                        submit(pool)

                    written_total += 1
                    rec.update({
                        'id': 'distill-v3-%06d' % written_total,
                        'dedup_key': key,
                        'category': cat,
                        'grid_index': idx,
                        'source': '%s-distillation-v3' % TEACHER_MODEL,
                        'synthetic': True,
                        'teacher_model': TEACHER_MODEL,
                        'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                    })
                    sink.write(json.dumps(rec, ensure_ascii=False) + '\n')
                    state[cat] += 1
                    accepted += 1
                    write_progress(cursor, state)

                    if accepted % 20 == 0:
                        flush()
                    if accepted % MIRROR_EVERY == 0:
                        flush(mirror=True)
                        print('  %d/%d accepted (mirrored)' % (accepted, len(jobs)))
                    if time.time() - last_flush > 30:
                        flush()
                        last_flush = time.time()

        flush(mirror=True)

    print()
    print('accepted   :', accepted)
    print('duplicates :', duplicates)
    print('overflow   :', overflow, '(accepted past quota by the headroom, discarded)')
    print('rejects    :', sum(rejects.values()),
          dict(sorted(rejects.items(), key=lambda kv: -kv[1])))
    for cat, reason, snippet in examples:
        print('  reject [%s] %s :: %s' % (cat, reason, snippet))
    print('total on disk:', written_total)
    print()
    short = [c for c, _, _ in CATEGORIES if state[c] < QUOTAS[c]]
    for name, _, _ in CATEGORIES:
        print('  %-30s %5d / %5d' % (name, state[name], QUOTAS[name]))
    if short:
        print()
        print('still short after headroom (re-run to top up):', ', '.join(short))

## Stage 2c — Audit what landed

Schema, enums, dedup, and the row-level split-disjointness proof. Read the output before
training: a corpus that fails here will fail silently later.


In [ ]:
import os, json, hashlib, subprocess
from collections import Counter

REPO = '/content/astra'
if not os.path.isdir(os.path.join(REPO, 'tools')):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git %s' % REPO,
                   shell=True, check=True)
os.chdir(REPO)

CLASSIFICATIONS = ('normal', 'suspicious', 'malicious', 'unknown')
RISK_LEVELS = ('INFO', 'LOW', 'MEDIUM', 'HIGH', 'CRITICAL')
HARDNESS_BY_RISK = {'INFO': 5, 'LOW': 20, 'MEDIUM': 45, 'HIGH': 70, 'CRITICAL': 90}
REQUIRED_FIELDS = ('input', 'classification', 'risk', 'early_warning',
                   'evidence', 'analysis', 'recommended_action', 'confidence')

# V1 drifted: 3 of 1002 records carried an extra "action" key. Anything outside this set is a
# schema change, not data, and is reported rather than silently carried into the corpus.
KNOWN_FIELDS = frozenset(REQUIRED_FIELDS) | {
    'id', 'dedup_key', 'category', 'grid_index', 'source', 'synthetic',
    'teacher_model', 'created_at', 'hardness'}


MIN_INPUT_CHARS, MIN_ANALYSIS_CHARS = 120, 40
# a 1200-character input plus its target comfortably fits and a runaway one provably does not.
# Capping at generation time is far cheaper than discovering 5000 over-long rows in the audit.
MAX_INPUT_CHARS, MAX_ANALYSIS_CHARS = 480, 420
# A teacher that loops will happily emit the same six events fifty times. That is valid JSON,
# passes every enum check and is worse than useless as training data, so reject it: if any
# 60-character span of the row occurs three or more times, the generation degenerated.
# Calibrated against every corpus in datasets/ - zero false positives over 7958 chat blocks.
REPEAT_ANCHOR, MAX_REPEATS = 60, 3
# A flag the model can actually learn: anything actionable is an early warning.
EARLY_WARNING_CLASSIFICATIONS = ('suspicious', 'malicious')
MAX_CONFIDENCE = 0.97
MIN_ACTION_CHARS = 40
# Phrases that assert there is nothing to do. Measured on V1: 122 of 1002 targets were under
# 40 characters and the single most common was "No action required." 93 times.
NO_OP_ACTIONS = ('no action required', 'no action needed', 'none', 'no action',
                 'no further action', 'nothing', 'n/a', 'not applicable')
CATEGORIES = ['Normal behavior', 'Suspicious behavior', 'Early-warning sequences',
              'False positives', 'Multi-signal correlation', 'Authentication',
              'Network', 'Process behavior', 'Vulnerability / code security']

RECORDS = 'datasets/distillation_v3/records.jsonl'
assert os.path.exists(RECORDS), 'no records.jsonl - run the generation cell first'
records = [json.loads(l) for l in open(RECORDS, encoding='utf-8') if l.strip()]
print('auditing %d records' % len(records))
print()

def is_no_op(action):
    """True when the action asserts that nothing needs to happen.

    V1 emitted "No action required." 93 times, "None." 18 times and several padded variants.
    Training on those teaches the model that the useful answer is sometimes nothing at all.
    """
    body = action.strip().rstrip('.!? ').lower()
    if body in NO_OP_ACTIONS:
        return True
    # also catch a padded form such as "No action required. This is standard telemetry."
    return any(body.startswith(phrase) and len(body) < len(phrase) + 90
               for phrase in NO_OP_ACTIONS)


def is_repetitive(text, anchor=REPEAT_ANCHOR, max_repeats=MAX_REPEATS, window=MAX_INPUT_CHARS):
    """The generator's span test, re-run here against what actually landed on disk."""
    head = text[:window]
    if len(head) < 4 * anchor:
        return False
    for off in (0, len(head) // 2, len(head) - 2 * anchor):
        frag = head[off:off + anchor]
        if len(frag) < anchor or not frag.strip():
            continue
        if head.count(frag) >= max_repeats:
            return True
    return False


faults = Counter()
accepted = Counter()
keys, examples = set(), []
for rec in records:
    rid = rec.get('id', '?')
    faults_missing = [f for f in REQUIRED_FIELDS if f not in rec]
    if faults_missing:
        faults['missing_fields'] += 1
    if rec.get('classification') not in CLASSIFICATIONS:
        faults['bad_classification'] += 1
    if rec.get('risk') not in RISK_LEVELS:
        faults['bad_risk'] += 1
    if not 0.0 <= float(rec.get('confidence', -1)) <= 1.0:
        faults['bad_confidence'] += 1
    if not isinstance(rec.get('early_warning'), bool):
        faults['early_warning_not_bool'] += 1
    if rec.get('hardness') != HARDNESS_BY_RISK.get(rec.get('risk')):
        faults['hardness_mismatch'] += 1
    if not isinstance(rec.get('evidence'), list) or not rec['evidence']:
        faults['evidence_empty'] += 1
    if '\n' in str(rec.get('input', '')):
        faults['input_multiline'] += 1
    if len(str(rec.get('input', ''))) < MIN_INPUT_CHARS:
        faults['input_too_short'] += 1
    if len(str(rec.get('input', ''))) > MAX_INPUT_CHARS:
        faults['input_too_long'] += 1
    if is_repetitive(str(rec.get('input', ''))):
        faults['input_repetitive'] += 1
    if len(str(rec.get('analysis', ''))) < MIN_ANALYSIS_CHARS:
        faults['analysis_too_short'] += 1
    if len(str(rec.get('analysis', ''))) > MAX_ANALYSIS_CHARS:
        faults['analysis_too_long'] += 1
    if is_repetitive(str(rec.get('analysis', ''))):
        faults['analysis_repetitive'] += 1
    # A no-op target is only a fault when an action is actually owed. Generated rows never
    # carry one (the prompt bans it), but salvage_v1.py keeps them on `normal` rows, because
    # that is the correct answer there and rejecting them deletes the entire normal class.
    # Counted separately so the deviation is visible rather than silently tolerated.
    if is_no_op(str(rec.get('recommended_action', ''))):
        if rec.get('classification') in EARLY_WARNING_CLASSIFICATIONS:
            faults['action_is_no_op'] += 1
        elif rec.get('classification') == 'normal':
            accepted['action_is_no_op_on_normal'] += 1
    if len(str(rec.get('recommended_action', ''))) < MIN_ACTION_CHARS:
        faults['action_too_short'] += 1
    if bool(rec.get('early_warning')) != (rec.get('classification') in EARLY_WARNING_CLASSIFICATIONS):
        faults['early_warning_inconsistent'] += 1
    if float(rec.get('confidence') or 0) > MAX_CONFIDENCE:
        faults['confidence_at_ceiling'] += 1
    for k in rec:
        if k not in KNOWN_FIELDS:
            faults['unexpected_field:' + k] += 1
    if rec.get('category') not in CATEGORIES:
        faults['unknown_category'] += 1
    if rec.get('synthetic') is not True or not rec.get('source'):
        faults['missing_provenance'] += 1
    expect = 'distill:' + hashlib.sha256(
        ' '.join(str(rec.get('input', '')).lower().split()).encode('utf-8')).hexdigest()[:16]
    if rec.get('dedup_key') != expect:
        faults['dedup_key_mismatch'] += 1
    if rec.get('dedup_key') in keys:
        faults['duplicate_record'] += 1
    keys.add(rec.get('dedup_key'))
    if rec.get('early_warning') is True and rec.get('classification') == 'normal':
        faults['early_warning_on_normal'] += 1
        if len(examples) < 3:
            examples.append(rid)

print('invariant violations:')
if faults:
    for name, n in faults.most_common():
        print('  %-26s %5d  (%.2f%%)' % (name, n, 100.0 * n / max(1, len(records))))
else:
    print('  none - every record satisfies every invariant')
if accepted:
    print()
    print('accepted deviations (reported, not counted as faults):')
    for name, n in accepted.most_common():
        print('  %-26s %5d  (%.2f%%)'
              % (name, n, 100.0 * n / max(1, len(records))))
print()

print('classification :', dict(Counter(r['classification'] for r in records).most_common()))
print('risk           :', dict(Counter(r['risk'] for r in records).most_common()))
print('category       :', dict(Counter(r['category'] for r in records).most_common()))
print()
print('early_warning rate : %.1f%%'
      % (100.0 * sum(1 for r in records if r['early_warning']) / max(1, len(records))))
print('mean confidence    : %.3f'
      % (sum(float(r['confidence']) for r in records) / max(1, len(records))))
print('mean input chars   : %.0f'
      % (sum(len(r['input']) for r in records) / max(1, len(records))))
print('duplicate keys     : %d' % (len(records) - len(keys)))
if examples:
    print()
    print('early_warning=true on a normal row, e.g.:', ', '.join(examples))
    print('these are legitimate (early signal on benign-looking telemetry) but worth eyeballing')

## Stage 3 — Class balance, splits, manifest

**Class cap first.** Any single `classification` is capped at `MAX_CLASS_SHARE` of the
corpus, with a floor of `MIN_PER_CLASS` so capping can never starve a class out of the
training split. Dropping is deterministic — sorted on `dedup_key`, then a seeded sample — so
the same corpus always balances the same way. The before/after counts and everything dropped
go into `manifest.json` under `class_balance`, so the corpus is self-describing.

**`records.jsonl` keeps every paid row, including the ones the cap dropped.** The cap is
applied to the in-memory corpus that becomes the splits, not to the file. That is deliberate:
the teacher rows are the expensive part, so keeping them means you can re-balance later with
a different `MAX_CLASS_SHARE` without paying for generation again. Read the class balance
from `manifest.json` or from the split files — `records.jsonl` is the uncapped input, and
counting classes there will show the skew the cap exists to correct. Expect roughly 40% of
paid rows to be discarded at the v2 skew of 67/21/12; `TARGET_SAMPLES` is 8000 to leave about
4900 balanced rows, and raising it is the only way to get more.

Then stratified splits, and the row-level disjointness proof: the split unit is one teacher
record, so a leaked record would show up as a shared body across two splits. That count must
be zero in every pair.


In [ ]:
import os, json, time, random, hashlib, shutil, subprocess
from collections import Counter

REPO = '/content/astra'
if not os.path.isdir(os.path.join(REPO, 'tools')):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git %s' % REPO,
                   shell=True, check=True)
os.chdir(REPO)

DATA_DIR = 'datasets/distillation_v3'
RECORDS = os.path.join(DATA_DIR, 'records.jsonl')
DRIVE_ROOT = '/content/drive/MyDrive/astra/distillation_v3'
SEED, VAL_FRAC, EVAL_FRAC = 20051, 0.01, 0.04
MIN_INPUT_CHARS, MIN_ANALYSIS_CHARS = 120, 40
# These caps bound a row. They do NOT make an exchange fit the 256-token block, and the
# comment that used to claim they did was wrong. Measured on the salvaged corpus at
# 3.17 chars/token, a 256-token block holds ~812 chars for the input AND the target
# together, while 480 + ~500 leaves ~980. 31% of blocks still overflow. Tightening input
# alone cannot fix it - at input<=240 it is still 6.6% over, because the target is the
# binding side. What the cap does buy is a worst case: max block fell 421 -> 380 tokens
# versus V1's 1200 cap, and a runaway teacher is rejected at generation time rather than
# discovered in the audit. The pre-flight cell reports the real distribution; read it
# before assuming a corpus fits.
MAX_INPUT_CHARS, MAX_ANALYSIS_CHARS = 480, 420
# A teacher that loops will happily emit the same six events fifty times. That is valid JSON,
# passes every enum check and is worse than useless as training data, so reject it: if any
# 60-character span of the row occurs three or more times, the generation degenerated.
# Calibrated against every corpus in datasets/ - zero false positives over 7958 chat blocks.
REPEAT_ANCHOR, MAX_REPEATS = 60, 3
# A flag the model can actually learn: anything actionable is an early warning.
EARLY_WARNING_CLASSIFICATIONS = ('suspicious', 'malicious')
MAX_CONFIDENCE = 0.97
MIN_ACTION_CHARS = 40
# Phrases that assert there is nothing to do. Measured on V1: 122 of 1002 targets were under
# 40 characters and the single most common was "No action required." 93 times.
NO_OP_ACTIONS = ('no action required', 'no action needed', 'none', 'no action',
                 'no further action', 'nothing', 'n/a', 'not applicable')

CATEGORIES = ['Normal behavior', 'Suspicious behavior', 'Early-warning sequences',
              'False positives', 'Multi-signal correlation', 'Authentication',
              'Network', 'Process behavior', 'Vulnerability / code security']

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_OK = os.path.isdir(DRIVE_ROOT)
except Exception:
    DRIVE_OK = False

assert os.path.exists(RECORDS), 'no records.jsonl - run the generation cell first'
records = [json.loads(l) for l in open(RECORDS, encoding='utf-8') if l.strip()]
assert records, 'records.jsonl is empty'
print('loaded %d records' % len(records))


def to_chat_block(rec):
    """datasets/chat/make_corpus.py:273 format."""
    return 'You: %s\nAstra: %s Classification: %s, risk %s. Recommended action: %s' % (
        rec['input'], rec['analysis'], rec['classification'].upper(),
        rec['risk'], rec['recommended_action'])


def to_alert_line(rec, row_id):
    """tools/build_cyber_corpus.py:52 shape, prose telemetry instead of 41 NSL-KDD features."""
    return 'ROW#%d ALERT %s | EARLY_WARNING %s RISK=%s HARDNESS=%d\n' % (
        row_id, rec['input'], rec['classification'], rec['risk'], rec['hardness'])


# ---- class balance cap -------------------------------------------------------
# Applied here, after generation, because it is the only point where the realised
# distribution is observable. The category weights in Stage 1 look like they should do this
# and do not: they are prompt archetypes, and the teacher picks the label.
#
# Two things this gets wrong if written naively, both measured on real data:
#
#  * A one-shot threshold does not hold. Capping against the ORIGINAL total leaves the
#    result above the target, because deleting rows lowers the total and the target with
#    it. 562 rows at 67% suspicious capped once to 253 leaves 57.6% against a 45% target.
#  * Iterating to a fixed point overshoots into the floor. As the total shrinks the cap
#    shrinks with it, and the MIN_PER_CLASS floor then dominates: a 995-row corpus with a
#    5-row rare class converged to 47% by cutting the two large classes to 40 each, keeping
#    85 of 995 rows. A single-class corpus collapsed 500 -> 40, which is data loss, not
#    balance - with one class there is no skew to correct.
#
# So: solve for the largest cap that satisfies the constraint, by binary search, and treat
# "no cap satisfies it" as "do not touch the data". _share() is monotone in K, so the
# search is exact rather than iterative.
def _capped(counts, K, floor):
    return {c: min(n, max(floor, K)) for c, n in counts.items()}


def _share(counts, K, floor):
    capped = _capped(counts, K, floor)
    return max(capped.values()) / max(1, sum(capped.values()))


before = Counter(r['classification'] for r in records)
dropped = Counter()
after = Counter()
n_before = sum(before.values())
cap = None
if len(before) > 1:
    lo, hi = 0, max(before.values())
    while lo <= hi:
        mid = (lo + hi) // 2
        if _share(before, mid, MIN_PER_CLASS) <= MAX_CLASS_SHARE + 1e-12:
            cap = mid
            lo = mid + 1
        else:
            hi = mid - 1

if cap is None:
    print('class balance: no cap satisfies %.0f%% for this distribution - left alone'
          % (MAX_CLASS_SHARE * 100))
    print('  a corpus with a single classification has no skew to correct.')
    print('  one whose rare classes are already at the MIN_PER_CLASS floor cannot be')
    print('  balanced by discarding data. Cutting rows here would lose training signal.')
    after = Counter(before)
    records = list(records)
else:
    rng = random.Random(SEED)
    by_class = {}
    for rec in records:
        by_class.setdefault(rec['classification'], []).append(rec)
    for items in by_class.values():
        items.sort(key=lambda r: r['dedup_key'])
    keep = {}
    for cls, items in sorted(by_class.items()):
        quota = min(len(items), max(MIN_PER_CLASS, cap))
        if quota >= len(items):
            keep[cls] = list(range(len(items)))
        else:
            order = list(range(len(items)))
            rng.shuffle(order)          # unbiased, and seeded so it repeats exactly
            keep[cls] = sorted(order[:quota])
        dropped[cls] = len(items) - len(keep[cls])
    records = [rec for c in by_class for i, rec in enumerate(by_class[c]) if i in keep[c]]
    after = Counter(r['classification'] for r in records)
    top = max(after.values()) / max(1, len(records))
    print('class balance: cap %d rows, max class %.1f%% of %d (was %.1f%% of %d), target %.0f%%'
          % (cap, top * 100, len(records),
             max(before.values()) / max(1, n_before) * 100, n_before, MAX_CLASS_SHARE * 100))
    for cls in sorted(set(before) | set(after)):
        print('  %-11s %5d -> %5d  (%5.1f%% -> %5.1f%%)'
              % (cls, before[cls], after[cls],
                 100.0 * before[cls] / n_before, 100.0 * after[cls] / max(1, len(records))))
    if dropped:
        print('  dropped %d over-represented rows: %s'
              % (sum(dropped.values()), {k: v for k, v in sorted(dropped.items()) if v}))
    assert records, 'the class cap removed every record'

by_cat = {}
for rec in records:
    by_cat.setdefault(rec['category'], []).append(rec)

splits = {'train': [], 'val': [], 'eval': []}
for cat, items in by_cat.items():
    items = sorted(items, key=lambda r: r['dedup_key'])
    n = len(items)
    n_val = min(n - 2, max(1, round(n * VAL_FRAC))) if n >= 3 else 0
    n_eval = (min(n - 2 - n_val, max(1, round(n * EVAL_FRAC)))
              if n - 2 - n_val >= 1 else 0)
    ordered = items[n_val + n_eval:] + items[:n_val] + items[n_val:n_val + n_eval]
    splits['train'].extend(ordered[: n - n_val - n_eval])
    splits['val'].extend(ordered[n - n_val - n_eval: n - n_eval])
    splits['eval'].extend(ordered[n - n_eval:])

for name in splits:
    splits[name].sort(key=lambda r: (r['category'], r['dedup_key']))

CHAT_DIR, ALERT_DIR = os.path.join(DATA_DIR, 'chat'), os.path.join(DATA_DIR, 'alerts')
os.makedirs(CHAT_DIR, exist_ok=True)
os.makedirs(ALERT_DIR, exist_ok=True)

split_meta = {'chat': {}, 'alerts': {}}
for name, items in splits.items():
    chat_path = os.path.join(CHAT_DIR, name + '.txt')
    alert_path = os.path.join(ALERT_DIR, name + '.txt')
    with open(chat_path, 'w', encoding='utf-8') as fh:
        # datasets/chat/make_corpus.py writes "\n\n".join(blocks) with no trailing newline;
        # match it exactly so both corpora are byte-comparable
        fh.write('\n\n'.join(to_chat_block(r) for r in items))
    with open(alert_path, 'w', encoding='utf-8') as fh:
        # tools/build_cyber_corpus.py writes one "\n"-terminated line per row
        for i, rec in enumerate(items):
            fh.write(to_alert_line(rec, i))
    dist = dict(Counter(r['classification'] for r in items).most_common())
    for flavour, path in (('chat', chat_path), ('alerts', alert_path)):
        blob = open(path, 'rb').read()
        split_meta[flavour][name] = {
            'path': path,
            'sha256': hashlib.sha256(blob).hexdigest(),
            'bytes': len(blob),
            'rows': len(items),
            'classification_distribution': dist,
        }
    print('%-6s %5d rows -> chat + alerts' % (name, len(items)))

bodies = {name: {r['dedup_key'] for r in items} for name, items in splits.items()}
shared = {'%s|%s' % (a, b): len(bodies[a] & bodies[b])
          for a, b in (('train', 'val'), ('train', 'eval'), ('val', 'eval'))}
print('shared record bodies across splits:', shared)
assert shared['train|val'] == 0 and shared['train|eval'] == 0, 'record body leaked across splits'

manifest = {
    'name': 'astra-distillation-v2',
    'kind': 'synthetic',
    'pipeline': 'colab/Distliation/astra_v1_distillation_v3.ipynb',
    'source_ids': ['gemini-3.5-flash-lite'],
    'teacher': 'gemini-3.5-flash-lite (Gemini API, GA 2026-07-21)',
    'teacher_model': 'gemini-3.5-flash-lite',
    'license': 'synthetic - generated output, no third-party corpus redistributed',
    'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'documents': len(records),
    'category_distribution': {c: len(by_cat.get(c, [])) for c in CATEGORIES},
    'dedup_redundancy_ratio': round(
        1.0 - len(bodies['train']) / max(1, len(splits['train'])), 6),
    'contamination_checked': True,
    'cleaning_rules': [
        'json-strict-parse: fenced or truncated replies rejected, never written',
        'schema-validate: all 8 required fields present and typed',
        'enum-check: classification in {normal,suspicious,malicious,unknown}',
        'risk-uppercase: risk in {INFO,LOW,MEDIUM,HIGH,CRITICAL} per configs/astra_200m.json',
        'single-line: multi-line teacher fields collapsed so the chat framing stays one turn per line',
        'dedup: sha256 of normalized input, global across all records',
        'min-length: input >= 120 chars, analysis >= 40 chars',
        'hardness-derived: hardness computed from risk, never taken from the teacher',
    ],
    'filters': [
        {'id': 'distill-v2-validator',
         'params': {'min_input_chars': MIN_INPUT_CHARS,
                    'min_analysis_chars': MIN_ANALYSIS_CHARS}},
        {'id': 'distill-v2-dedup', 'params': {'key': 'sha256(normalized input)[:16]'}},
    ],
    'hash': {'algorithm': 'sha256', 'value': split_meta['chat']['train']['sha256']},
    'row_level_safety': {
        'shared_record_bodies_across_splits': shared,
        'note': 'the split unit is one teacher record; the n=13 token gate is reported '
                'separately by the pre-flight cell because the You:/Astra: framing template '
                'repeats in every chat block',
    },
    'chat_format': 'You: <question> / Astra: <answer>, blank-line separated '
                   '(datasets/chat/make_corpus.py:273)',
    'alerts_format': 'ROW#<n> ALERT <telemetry> | EARLY_WARNING <classification> '
                     'RISK=<RISK> HARDNESS=<n> - shape from tools/build_cyber_corpus.py:52 with '
                     'prose telemetry instead of 41 NSL-KDD features; never concatenate with '
                     'datasets/cyber/',
    'splits': split_meta,
}

manifest_path = os.path.join(DATA_DIR, 'manifest.json')
manifest['class_balance'] = {
    'max_share': MAX_CLASS_SHARE,
    'before': dict(before),
    'after': dict(after),
    'dropped': dict(dropped),
}
with open(manifest_path, 'w', encoding='utf-8') as fh:
    json.dump(manifest, fh, indent=2, sort_keys=True)
    fh.write('\n')
print()
print('manifest ->', manifest_path)
for flavour in split_meta:
    for name in split_meta[flavour]:
        m = split_meta[flavour][name]
        print('  %-7s %-5s %5d rows %10d bytes sha=%s'
              % (flavour, name, m['rows'], m['bytes'], m['sha256'][:16]))

if DRIVE_OK:
    # each split keeps its own Drive subdirectory: chat/ and alerts/ both contain
    # train.txt, val.txt and eval.txt, so mirroring them flat would overwrite one with the other
    for rel in ('manifest.json',
                'chat/train.txt', 'chat/val.txt', 'chat/eval.txt',
                'alerts/train.txt', 'alerts/val.txt', 'alerts/eval.txt'):
        src = manifest_path if rel == 'manifest.json' else os.path.join(DATA_DIR, rel)
        if not os.path.exists(src):
            continue
        dst = os.path.join(DRIVE_ROOT, rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
    print('mirrored manifest + splits to Drive ->', DRIVE_ROOT)
    for rel in ('chat/train.txt', 'alerts/train.txt'):
        print('  %-18s %10d bytes'
              % (rel, os.path.getsize(os.path.join(DRIVE_ROOT, rel))))

print()
print('first train chat block:')
print(open(os.path.join(CHAT_DIR, 'train.txt'), encoding='utf-8').read().split('\n\n')[0])
print()
print('first train alert line:')
print(open(os.path.join(ALERT_DIR, 'train.txt'), encoding='utf-8').readline().strip())

## Stage 4 — Emit real training configs

Writes `configs/astra_distill_v3_{chat,alerts}.json` with paths that point at the corpora
just written. `leak_check` is off with a reason: the n=13 token gate compares raw token
n-grams, and every block repeats the `You:`/`Astra:` framing, so it measures the template
rather than contamination. The row-level proof above is the real gate.


In [ ]:
import os, json, shutil, subprocess

REPO = '/content/astra'
if not os.path.isdir(os.path.join(REPO, 'tools')):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git %s' % REPO,
                   shell=True, check=True)
os.chdir(REPO)

DATA_DIR = 'datasets/distillation_v3'
TOK = 'tokenizer/artifacts/prose_chat_word.json'
DRIVE_ROOT = '/content/drive/MyDrive/astra/distillation_v3'
SEED = 20051

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_OK = os.path.isdir(DRIVE_ROOT)
except Exception:
    DRIVE_OK = False

MODEL_BLOCK = {
    'vocab_size': 16384,
    'd_model': 2048,
    'n_layers': 24,
    'n_heads': 16,
    'd_head': 128,
    'd_ffn': 8192,
    'max_seq_len': 512,
    'rope_theta': 10000.0,
    'eps': 1e-06,
    'tie_embeddings': True,
}

TRAINING_BLOCK = {
    'max_steps': 6000, 'peak_lr': 3e-4, 'min_lr': 1e-6, 'warmup_steps': 60,
    'batch_seq': 4, 'accum_steps': 2, 'optimizer': 'adamw', 'scheduler': 'cosine',
    'weight_decay': 0.05, 'grad_clip': 1.0, 'val_every': 200, 'val_shards': 1,
    'save_every': 2000,  # 1.64B snapshot is 3.3 GB weights-only; 100 would ask for hundreds of GB on a ~78 GB Colab disk
    'experiment_store': 'experiments/runs',
}

LEAK_NOTE = (
    'n-gram gate disabled for the same reason as configs/astra5m_word_chat.json: with a '
    'word-level tokenizer 13 tokens is a ~13-word window, and the You:/Astra: framing plus a '
    'shared risk vocabulary repeats across every block, so the gate reports overlap on a '
    'corpus with zero record-level leakage. Record-level disjointness is proven in '
    'datasets/distillation_v3/manifest.json (row_level_safety) and the raw n=13 numbers are '
    'printed by the pre-flight cell.'
)

CONFIGS = {
    'configs/astra_distill_v3_chat.json': {
        'data': {'train': DATA_DIR + '/chat/train.txt', 'val': DATA_DIR + '/chat/val.txt'},
        'out_dir': 'checkpoints/astra_distill_v3_chat',
    },
    'configs/astra_distill_v3_alerts.json': {
        'data': {'train': DATA_DIR + '/alerts/train.txt', 'val': DATA_DIR + '/alerts/val.txt'},
        'out_dir': 'checkpoints/astra_distill_v3_alerts',
    },
}

os.makedirs('configs', exist_ok=True)
for path, spec in CONFIGS.items():
    for key in ('train', 'val'):
        assert os.path.exists(spec['data'][key]), 'missing corpus: ' + spec['data'][key]
    cfg = {'model': MODEL_BLOCK, 'training': TRAINING_BLOCK, 'tokenizer': TOK,
           'data': spec['data'], 'seed': SEED,
           'safety': {'leak_check': False, 'note': LEAK_NOTE},
           'out_dir': spec['out_dir']}
    with open(path, 'w', encoding='utf-8') as fh:
        json.dump(cfg, fh, indent=2)
        fh.write('\n')
    print('wrote %s\n      train %s\n      val   %s' % (path, spec['data']['train'], spec['data']['val']))

if DRIVE_OK:
    for path in CONFIGS:
        shutil.copy2(path, os.path.join(DRIVE_ROOT, os.path.basename(path)))
    print('configs mirrored to Drive ->', DRIVE_ROOT)

## Stage 4b — Restore configs from Drive

Useful when resuming in a fresh runtime: the corpus is on Drive but `/content` was reset.


In [ ]:
import os, shutil, subprocess

REPO = '/content/astra'
if not os.path.isdir(os.path.join(REPO, 'tools')):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git %s' % REPO,
                   shell=True, check=True)
os.chdir(REPO)

# Keep the Drive copy of the manifest current: Stage 6 appends to it, and a Drive manifest
# that predates the pre-flight is what makes a later audit disagree with the local file.
MANIFEST = 'datasets/distillation_v3/manifest.json'
DRIVE_ROOT = '/content/drive/MyDrive/astra/distillation_v3'
if os.path.exists(MANIFEST) and os.path.isdir(DRIVE_ROOT):
    shutil.copy2(MANIFEST, os.path.join(DRIVE_ROOT, 'manifest.json'))
    print('refreshed Drive manifest ->', os.path.join(DRIVE_ROOT, 'manifest.json'))
else:
    print('manifest refresh skipped (no manifest, or Drive not mounted)')

## Stage 5 — Pre-flight: tokenizer, coverage, block lengths, leak report

Rebuilds the tokenizer with `--extra` if it is missing (it is gitignored, so a fresh clone
has none) and reports what actually matters:

* **OOV rate.** The number that decides whether a 6-layer model can learn this text at all.
* **Block-length distribution.** The student's block is 256 tokens, so an exchange that does
  not fit is split and the model never sees the input and the answer in the same context.
  This is **reported, not enforced**, and the caps in Stage 1 do not fix it — the target is
  the binding side, not the telemetry. Read the numbers before assuming the corpus fits.
* **Leak report.** The figure the disabled gate would have tripped on.


In [ ]:
import os, sys, json, time, subprocess

REPO = '/content/astra'
if not os.path.isdir(os.path.join(REPO, 'tools')):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git %s' % REPO,
                   shell=True, check=True)
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, 'python'))

TOK_ARTIFACT = 'tokenizer/artifacts/prose_chat_word.json'
if not os.path.exists(TOK_ARTIFACT):
    print('tokenizer artifact missing (gitignored) - building it')
    # --extra unions the shipped corpora into the vocab. Without it this corpus OOVs at
    # ~16%, which a 6-layer model cannot learn through.
    subprocess.run('python tools/build_word_tokenizer.py --extra', shell=True, check=True)
assert os.path.exists(TOK_ARTIFACT), 'tokenizer artifact still missing'

from astra.safety import leak_check
from astra.tokenizer.word import WordLevel

tok = WordLevel.load(TOK_ARTIFACT)
unk_id = tok.special_names.index('<unk>')
print('tokenizer: %d tokens (%d words), min_frequency=%d, unk_id=%d'
      % (len(tok), tok.stats.get('num_words', 0), tok.min_frequency, unk_id))
print()

DATA_DIR = 'datasets/distillation_v3'
train_text, val_text = {}, {}
for corpus, store in (('chat/train.txt', train_text), ('chat/val.txt', val_text),
                      ('alerts/train.txt', train_text), ('alerts/val.txt', val_text)):
    path = os.path.join(DATA_DIR, corpus)
    assert os.path.exists(path), 'missing corpus: ' + path
    store[corpus] = open(path, encoding='utf-8').read()

for corpus, text in train_text.items():
    ids = tok.encode(text)
    oov = sum(1 for i in ids if i == unk_id)
    print('%-16s %8d chars %8d tokens  OOV=%5.2f%%  bytes/token=%.1f'
          % (corpus, len(text), len(ids), 100.0 * oov / max(1, len(ids)),
             len(text) / max(1, len(ids))))
print()

# Block-length report. The student's block is 256 tokens, so an exchange that does not fit is
# split and the model never sees the input and the answer in the same context. This is REPORTED,
# not enforced, and the reason is that the repo already tolerates it: datasets/chat sits at 49.8%
# over 256. Do not read a high number here as a bug to be fixed by lowering the input cap - on
# the salvaged corpus, input<=240 still leaves 6.6% of blocks over 256, because the target is the
# binding side. To actually fit 256 you have to shorten analysis and recommended_action, not the
# telemetry. V1 (1200 cap) peaked at 421 tokens; the 480 cap brings that to 380.
BLOCK_TOKENS = 256
blocks = [b for b in train_text['chat/train.txt'].split('\n\n') if b.strip()]
blen = sorted(len(tok.encode(b)) for b in blocks)
nblocks = len(blen)
print('chat block length (student block = %d tokens, %d blocks)' % (BLOCK_TOKENS, nblocks))
print('  tokens min/p50/p90/p95/p99/max : %d / %d / %d / %d / %d / %d'
      % (blen[0], blen[nblocks // 2], blen[int(nblocks * .90)], blen[int(nblocks * .95)],
         blen[int(nblocks * .99)], blen[-1]))
for lim in (256, 384, 512):
    over = sum(1 for x in blen if x > lim)
    print('  over %3d tokens : %5d / %5d  (%5.1f%%)'
          % (lim, over, nblocks, 100.0 * over / max(1, nblocks)))
print()

result = leak_check(tok.encode(train_text['chat/train.txt']),
                    tok.encode(val_text['chat/val.txt']), n=13)
print('leak_check(chat train, chat val, n=13) =')
print(json.dumps(result, indent=2))
print()
print('That is the number the disabled safety gate would have tripped on. The gate is off')
print('because a repeated chat framing template makes any overlap meaningless here; the')
print('record-level proof lives in manifest.json under row_level_safety.')

manifest_path = os.path.join(DATA_DIR, 'manifest.json')
assert os.path.exists(manifest_path), 'no manifest.json - run the materialize cell first'
manifest = json.load(open(manifest_path, encoding='utf-8'))
manifest['leak_report'] = result
manifest['preflight'] = {
    'tokenizer': TOK_ARTIFACT,
    'tokenizer_vocab_size': len(tok),
    'block_tokens': BLOCK_TOKENS,
    'chat_block_tokens': {
        'count': nblocks, 'min': blen[0], 'p50': blen[nblocks // 2],
        'p90': blen[int(nblocks * .90)], 'p95': blen[int(nblocks * .95)],
        'p99': blen[int(nblocks * .99)], 'max': blen[-1],
        'over_256': sum(1 for x in blen if x > 256),
        'over_384': sum(1 for x in blen if x > 384),
    },
    'recorded_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
with open(manifest_path, 'w', encoding='utf-8') as fh:
    json.dump(manifest, fh, indent=2, sort_keys=True)
    fh.write('\n')
print()
print('leak_report + preflight recorded in', manifest_path)

## Stage 6a — Training

On by default: this is the stage that turns a corpus into a model. Two stages — **alerts
first** (sequence + label head), then **chat warm-started from the alerts checkpoint** with
`--reset-step` so the LR schedule and step counter restart rather than resuming mid-schedule.

**Set the runtime to a GPU first** (Runtime → Change runtime type → T4). The cell asserts on
`cuda.is_available()` rather than falling back to CPU, because a CPU run would look like a
hang rather than an error.

`bf16` is used whenever torch reports the GPU as supporting it; a T4 reports support via
emulation, so a T4 run uses bf16. The check stays for older torch and for GPUs that genuinely
lack it. No GradScaler is needed on the fp16 path because `gpu_train.py` does not scale the
loss.


In [ ]:
import glob

RUN_TRAIN = True
DTYPE = 'bf16'


def run_logged(cmd, log):
    """Run a training command, streaming its output live but keeping a log.

    `subprocess.run(shell=True, check=True)` is what hid the real error the first time this
    ran: the CalledProcessError carried the command and not the message, so the actual
    failure only existed in a scrollback nobody read. Stream it AND keep the tail."""
    print('\n$ ' + cmd, flush=True)
    with open(log, 'w', encoding='utf-8') as fh:
        proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            sys.stdout.write(line)
            fh.write(line)
    rc = proc.wait()
    if rc:
        tail = ''.join(open(log, encoding='utf-8').readlines()[-40:])
        raise SystemExit('%s\n\nfailed with exit %d. Last 40 lines of %s:\n%s'
                         % (cmd, rc, log, tail))


def find_final(out_dir):
    """gpu_train.py writes <out>/final.npz, or <out>/resumed/final.npz with --resume."""
    plain = os.path.join(out_dir, 'final.npz')
    if os.path.exists(plain):
        return plain
    hits = sorted(glob.glob(os.path.join(out_dir, '**', 'final.npz'), recursive=True))
    return hits[0] if hits else None


def persist(out_dir):
    if not DRIVE_OK:
        return
    dst = os.path.join(DRIVE_ROOT, 'runs', os.path.basename(out_dir))
    os.makedirs(dst, exist_ok=True)
    for src in [os.path.join(out_dir, n) for n in ('final.npz', 'report.json')] + \
            glob.glob(os.path.join(out_dir, '*.manifest.json')):
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(dst, os.path.basename(src)))


import torch
if not RUN_TRAIN:
    print('RUN_TRAIN is False - set it to True to spend GPU time.')
else:
    assert torch.cuda.is_available(), (
        'No GPU visible. Runtime -> Change runtime type -> T4, then re-run this cell. '
        'Falling back to CPU here would look like a hang rather than an error.')
    gpu = torch.cuda.get_device_name(0)
    print('torch %s  cuda=%s  %s' % (torch.__version__, torch.cuda.is_available(), gpu))
    if DTYPE == 'bf16' and not torch.cuda.is_bf16_supported():
        cap = torch.cuda.get_device_capability(0)
        print('%s has no native bfloat16 (sm_%d) - torch.amp.autocast raises on it, '
              'so using fp16' % (gpu, cap[0] * 10 + cap[1]))
        DTYPE = 'fp16'
    print('dtype:', DTYPE)

    out_alerts = os.path.join(RUNS, 'distill_v3_alerts')
    os.makedirs(out_alerts, exist_ok=True)
    run_logged('%s training/gpu_train.py '
               '--config configs/astra_distill_v3_alerts.json '
               '--out %s --cache-dir /tmp/astra_cache --dtype %s'
               % (sys.executable, out_alerts, DTYPE), 'train_alerts.log')
    alerts_final = find_final(out_alerts)
    assert alerts_final, 'alerts stage produced no final.npz'
    print('alerts final:', alerts_final)
    persist(out_alerts)

    assert os.path.exists(alerts_final), 'alerts final.npz vanished - rerun alerts stage'
    out_chat = os.path.join(RUNS, 'distill_v3_chat')
    os.makedirs(out_chat, exist_ok=True)
    run_logged('%s training/gpu_train.py '
               '--config configs/astra_distill_v3_chat.json '
               '--out %s --resume %s --reset-step '
               '--cache-dir /tmp/astra_cache --dtype %s'
               % (sys.executable, out_chat, alerts_final, DTYPE), 'train_chat.log')
    chat_final = find_final(out_chat)
    assert chat_final, 'chat stage produced no final.npz'
    print('chat final:', chat_final)
    persist(chat_final and out_chat)
    print('\ndone - checkpoints under %s'
          % (DRIVE_ROOT if DRIVE_OK else '(no Drive, so only on this VM)'))

## Stage 6b — Smoke test

A checkpoint that loads is not a checkpoint that learned something. Three prompts and a
held-out alert row, so you can see the output shape before trusting it. An empty reply or a
string of special tokens means the tokenizer and the checkpoint disagree — a real bug, not a
quality problem.


In [ ]:
import os, sys, glob, json, subprocess

import numpy as np

REPO = '/content/astra'
if not os.path.isdir(os.path.join(REPO, 'tools')):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git %s' % REPO,
                   shell=True, check=True)
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, 'python'))

from astra.inference import KVCache, decode
from astra.learning.evaluate import load_model
from astra.model import ModelConfig
from astra.tokenizer.word import WordLevel

candidates = sorted(glob.glob(os.path.join(RUNS, 'distill_v3_chat', '**', 'final.npz'), recursive=True))
assert candidates, 'no distilled chat checkpoint - set RUN_TRAIN = True and run Stage 6a'
CKPT = candidates[0]
print('checkpoint:', CKPT)

cfg = json.load(open('configs/astra_distill_v3_chat.json', encoding='utf-8'))
tok = WordLevel.load(cfg['tokenizer'])
mcfg = ModelConfig.from_dict(cfg['model'])
mcfg.vocab_size = len(tok)

model = load_model(mcfg, CKPT)
cache = KVCache(mcfg)
SPECIALS = set(range(tok.num_special))
history = []


def ask(user_text, max_new=64, temperature=0.7, top_k=40, rep_penalty=1.15):
    global history
    history.append('You: %s' % user_text)
    prompt = '\n'.join(history) + '\nAstra:'
    ids = tok.encode(prompt)[-(mcfg.max_seq_len - max_new):]
    new = decode(model, ids, max_new=max_new, temperature=temperature,
                 rng=np.random.default_rng(len(history)), top_k=top_k,
                 cache=cache, rep_penalty=rep_penalty, forbidden=SPECIALS)
    reply = tok.decode(new).split('\nYou:')[0].strip()
    history.append('Astra: %s' % reply)
    return reply


for q in ['an ssh login as root from 203.0.113.9 at 02:14 outside the change window',
          'explain what early warning means in a security alert stream',
          'what should an analyst do about a service account whose permissions widened']:
    print('You  :', q)
    print('Astra:', ask(q))
    print()

alert_path = 'datasets/distillation_v3/alerts/eval.txt'
if os.path.exists(alert_path):
    print('held-out alert row (never trained on):')
    print(' ', open(alert_path, encoding='utf-8').readline().strip()[:320])

## Stage 7 — Download

One zip, because the pieces are useless apart. The one people forget is the **tokenizer**:
`.gitignore` excludes `tokenizer/artifacts/`, so it is not in the repo and cannot be
re-cloned, and every config points at it. Without it nothing loads.

* `datasets/distillation_v3/**` — the corpora, `records.jsonl` (provenance) and `manifest.json`
  (every drop count, repair, the class-balance before/after, the leak report, preflight).
* `configs/astra_distill_v3_{chat,alerts}.json` — paths already point at the data above.
* `tokenizer/artifacts/prose_chat_word.json` — **required**, rebuilt with `--extra`.
* `runs/**/final.npz` — the checkpoints, only if `RUN_TRAIN` ran. This is the actual model.
* `runs/**/*.manifest.json` — step, seed, params, loss history. Not decoration:
  `load_checkpoint` reads them, and a warm-start without them silently restarts the schedule.

`SHA256SUMS.txt` is included so you can check the zip arrived intact.


In [ ]:
import glob, zipfile, hashlib

want = []
want += sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.txt'), recursive=True))
want += [os.path.join(DATA_DIR, 'records.jsonl'), os.path.join(DATA_DIR, 'manifest.json')]
want += ['configs/astra_distill_v3_chat.json', 'configs/astra_distill_v3_alerts.json']
want += [os.path.join(REPO, TOK_ARTIFACT)]
want += sorted(glob.glob(os.path.join(RUNS, '**', 'final.npz'), recursive=True))
want += sorted(glob.glob(os.path.join(RUNS, '**', 'report.json'), recursive=True))
want += sorted(glob.glob(os.path.join(RUNS, '**', '*.manifest.json'), recursive=True))

have = [p for p in want if os.path.exists(p)]
skipped = [os.path.basename(p) for p in want if not os.path.exists(p)]
if skipped:
    print('not produced, skipping:', skipped)
assert have, 'nothing to package - did the generation cell run?'


def arcname(path):
    """The name this file has INSIDE the zip. Checkpoints sit under RUNS, which is outside
    the repo, so they cannot be laid out relative to the repo without climbing out with '..'
    - and SHA256SUMS.txt has to list exactly these names or `sha256sum -c` cannot verify."""
    if os.path.abspath(path).startswith(os.path.abspath(RUNS) + os.sep):
        return os.path.join('runs', os.path.relpath(path, RUNS))
    return os.path.relpath(path, REPO)


arcs = {p: arcname(p) for p in have}
OUT = os.path.abspath('astra_distill_v3.zip')
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in have:
        z.write(p, arcs[p])
    z.writestr('SHA256SUMS.txt', '\n'.join(
        '%s  %s' % (hashlib.sha256(open(p, 'rb').read()).hexdigest(), arcs[p])
        for p in have) + '\n')

print('%-52s %10s' % ('file', 'bytes'))
for p in have:
    print('%-52s %10d' % (arcs[p], os.path.getsize(p)))
print('\n%d files, %.1f MB zipped' % (len(have), os.path.getsize(OUT) / 1e6))

if not DRIVE_OK:
    try:
        from google.colab import files
        print('\ndownloading', OUT)
        files.download(OUT)
    except ImportError:
        print('\nnot under Colab - the zip is at', OUT)
else:
    shutil.copy2(OUT, os.path.join(DRIVE_ROOT, os.path.basename(OUT)))
    print('\ncopied to Drive:', os.path.join(DRIVE_ROOT, os.path.basename(OUT)))